##### Ingest lap_times folder

##### Read the CSV file using the spark dataframe reader API

In [0]:
dbutils.widgets.text("p_data_source", "")
v_data_source = dbutils.widgets.get("p_data_source")

In [0]:
dbutils.widgets.text("p_file_date", "2021-03-28")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
lap_times_schema = StructType(fields=[StructField("raceId", IntegerType(), False),
                                      StructField("driverId", IntegerType(), True),
                                      StructField("lap", IntegerType(), True),
                                      StructField("position", IntegerType(), True),
                                      StructField("time", StringType(), True),
                                      StructField("milliseconds", IntegerType(), True)
                                     ])

In [0]:
lap_times_df = spark.read \
.schema(lap_times_schema) \
.csv(f"{raw_path}/lap_times")

##### Rename columns and add new columns

In [0]:
final_df = lap_times_df \
.withColumnRenamed("raceId", "race_id") \
.withColumnRenamed("driverId", "driver_id") \
.withColumn("data_source", lit(v_data_source))

In [0]:
final_df = add_ingestion_date(final_df)

##### Write to output to processed container in parquet format

In [0]:
### final solution 
overwrite_partition_free(final_df, 'f1_processed', 'lap_times', 'race_id')

In [0]:
# final_df.write.mode("overwrite").parquet(f"{processed_path}/lap_times")

##### Check

In [0]:
# spark.read.parquet(f"{processed_path}/lap_times").display()

In [0]:
dbutils.notebook.exit("SUCCESS")

In [0]:
%sql
select race_id, count(1) as count
from f1_processed.lap_times
group by race_id
order by race_id desc;